In [0]:
%python
VOLUME = "/Volumes/workspace/energy/raw_files"
FILE = "DE_generation.csv"

df = spark.read.csv(
    f"{VOLUME}/{FILE}",
    header=True,
    sep=";",
    encoding="UTF-8"
)

print(df.count(), "rows")
df.printSchema()

53112 rows
root
 |-- Datum von: string (nullable = true)
 |-- Datum bis: string (nullable = true)
 |-- Biomasse [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Wasserkraft [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Wind Offshore [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Wind Onshore [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Photovoltaik [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Sonstige Erneuerbare [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Kernenergie [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Braunkohle [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Steinkohle [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Erdgas [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Pumpspeicher [MWh] Berechnete Auflösungen: string (nullable = true)
 |-- Sonstige Konventionelle [MWh] Berechnete Auflösungen: string (nullable = true)



In [0]:
%python
import re

def clean(c):
    c = c.lower()
    c = c.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue")
    c = re.sub(r"\[.*?\]", "", c)                        # remove [MWh]
    c = re.sub(r"berechnete aufloesungen|originalaufloesungen", "", c)
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")      # spaces -> underscores

df = df.toDF(*[clean(c) for c in df.columns])
print(df.columns)

['datum_von', 'datum_bis', 'biomasse', 'wasserkraft', 'wind_offshore', 'wind_onshore', 'photovoltaik', 'sonstige_erneuerbare', 'kernenergie', 'braunkohle', 'steinkohle', 'erdgas', 'pumpspeicher', 'sonstige_konventionelle']


In [0]:
%python
from pyspark.sql import functions as F

bronze = (df
          .withColumn("ingested_at", F.current_timestamp())
          .withColumn("source_file", F.lit(FILE)))

(bronze.write
       .mode("overwrite")
       .saveAsTable("workspace.energy.bronze_generation"))

print("saved")

saved


In [0]:
%python
t = spark.table("workspace.energy.bronze_generation")
print(t.count(), "rows")
display(t.limit(5))

53112 rows


datum_von,datum_bis,biomasse,wasserkraft,wind_offshore,wind_onshore,photovoltaik,sonstige_erneuerbare,kernenergie,braunkohle,steinkohle,erdgas,pumpspeicher,sonstige_konventionelle,ingested_at,source_file
01.10.2020 00:00,01.10.2020 01:00,"4.160,50","1.930,75","5.396,75","7.087,75","0,00","177,00","6.626,50","12.365,50","3.330,25","7.329,50","1.404,75","1.515,25",2026-10-27T09:21:38.644Z,DE_generation.csv
01.10.2020 01:00,01.10.2020 02:00,"4.147,50","1.961,50","5.335,50","7.454,50","0,00","173,00","6.634,75","11.744,50","3.055,50","7.151,75","2.090,25","1.502,50",2026-10-27T09:21:38.644Z,DE_generation.csv
01.10.2020 02:00,01.10.2020 03:00,"4.136,00","1.923,25","4.910,50","7.704,75","0,00","178,00","6.644,25","10.848,50","2.863,00","6.601,50","2.127,50","1.515,50",2026-10-27T09:21:38.644Z,DE_generation.csv
01.10.2020 03:00,01.10.2020 04:00,"4.130,50","1.893,75","4.933,75","8.053,50","0,00","177,75","6.650,00","10.792,25","3.113,00","6.796,75","2.051,00","1.509,25",2026-10-27T09:21:38.644Z,DE_generation.csv
01.10.2020 04:00,01.10.2020 05:00,"4.149,50","1.936,25","4.663,50","8.306,50","0,00","178,50","6.656,00","11.396,75","3.261,75","6.971,25","2.030,50","1.509,00",2026-10-27T09:21:38.644Z,DE_generation.csv
